# Type Sinhala, hear the model

Run this **interactively** (open it and run cells), not with Save & Run All - the point is
to type your own text and listen.

It loads the **deployment candidate** (`model_fp16.pth`, 0.934 GB) from run 5's output and
gives you one function:

```python
say("ආයුබෝවන්, ඔබට කොහොමද?")                  # dinithi
say("මම අද උදේ පාසල් ගියා.", "harini")        # harini
say(text, ab=True)                             # fp16 vs fp32, same text, back to back
```

**Set Accelerator to GPU T4 x2 before running.** The API cannot set it, and the notebook
will stop at the preflight cell if it is anything else.

### What the objective numbers could not tell you

Run 5 measured MCD 62.8, F0 corr 0.44, SECS 0.71, UTMOS 2.70 against real recordings at
3.27. UTMOS is English-trained and under-rates Sinhala, so the *gap* is the signal and the
absolute number is not. **No metric here says whether this is good enough to ship** - that
is what your ears are for. Listen for:

- **words that are wrong** rather than merely accented - the transliteration is the usual cause
- **truncation** (a sentence that stops early) and **looping** (a syllable that repeats)
- **intonation** - the weakest measured axis, F0 corr 0.44, and worse for harini at 0.33

## 1. Install - restart the session after this cell

In [ ]:
!pip install -q "coqui-tts>=0.25.1" soundfile librosa

## 2. Preflight, code and model

In [ ]:
import torch

name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
if name is None:
    raise RuntimeError("no GPU. Settings -> Accelerator -> GPU T4 x2.")
arch = "sm_%d%d" % torch.cuda.get_device_capability(0)
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError("%s is %s; this torch has kernels for %s. "
                       "Settings -> Accelerator -> GPU T4 x2."
                       % (name, arch, " ".join(torch.cuda.get_arch_list())))
print("GPU:", name, arch)

In [ ]:
import glob, os, shutil, subprocess, sys, time

CODE = "/kaggle/temp/code"
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git",
                CODE], check=True)
sys.path.insert(0, CODE + "/xtts_sinhala")
from sinhala_text import to_ascii

# run 5's export, attached as an input
hits = glob.glob("/kaggle/input/**/model_fp16.pth", recursive=True)
if not hits:
    raise RuntimeError("Add Input -> Notebook Output -> uom230429e/xtts-new-optimised")
EXPORT = os.path.dirname(min(hits, key=len))
FP16, SLIM = EXPORT + "/model_fp16.pth", EXPORT + "/model_slim.pth"

# One reference clip per speaker, straight from the corpus -- no dataset build
# needed. XTTS resamples internally, so the 44.1 kHz originals are fine.
REFS = {}
for spk in ("dinithi", "harini"):
    wavs = sorted(glob.glob("/kaggle/input/**/*%s*/**/*.wav" % spk.capitalize(),
                            recursive=True))
    if not wavs:
        wavs = sorted(w for w in glob.glob("/kaggle/input/**/*.wav", recursive=True)
                      if spk[:3] in os.path.basename(w).lower())
    REFS[spk] = wavs[len(wavs) // 2] if wavs else None
    print("%-8s ref: %s" % (spk, REFS[spk]))

OUT = "/kaggle/working/samples"
os.makedirs(OUT, exist_ok=True)

## 3. Load the model and cache the speaker latents

The conditioning latents are computed **once per speaker** and reused for every sentence.
That is how this should be served too: there are exactly two speakers and they never
change, so recomputing them per request is wasted work on the critical path.

In [ ]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

def load(ckpt):
    cfg = XttsConfig()
    cfg.load_json(EXPORT + "/config.json")
    m = Xtts.init_from_config(cfg)
    m.load_checkpoint(cfg, checkpoint_path=ckpt,
                      vocab_path=EXPORT + "/vocab.json", use_deepspeed=False)
    m.cuda().eval()
    return m, cfg

t0 = time.time()
MODEL, CFG = load(FP16)
print("loaded %s in %.1f s" % (os.path.basename(FP16), time.time() - t0))

LATENTS = {}
for spk, ref in REFS.items():
    if ref:
        LATENTS[spk] = MODEL.get_conditioning_latents(
            audio_path=[ref], gpt_cond_len=CFG.gpt_cond_len,
            max_ref_length=CFG.max_ref_len, sound_norm_refs=CFG.sound_norm_refs)
print("cached latents for:", list(LATENTS))

## 4. `say()`

Decode settings are the ones session B validated: temperature 0.75, repetition_penalty 5.0,
top_k 50, top_p 0.85. Lowering the temperature was measured and **did not help** - it added
a failure and cost harini 0.04 of F0 correlation - so these are defaults worth keeping
unless your ears disagree.

Raw Sinhala is never fed to the model. `to_ascii()` runs first, and the ASCII it produces is
printed every time so you can see exactly what was spoken. A word that sounds wrong is
usually wrong *there*, not in the acoustics.

In [ ]:
import numpy as np, soundfile as sf
from IPython.display import Audio, display

def say(text, speaker="dinithi", model=None, temperature=0.75, repetition_penalty=5.0,
        top_k=50, top_p=0.85, split=False, save_as=None, ab=False, show=True):
    """Synthesise Sinhala (or already-romanised) text and play it."""
    if ab:
        return _ab(text, speaker, temperature, repetition_penalty, top_k, top_p, split)

    m = model or MODEL
    ascii_text = to_ascii(text)
    if show:
        print(text)
        print("  ->", ascii_text)

    lat, emb = LATENTS[speaker]
    t0 = time.time()
    out = m.inference(text=ascii_text, language="en", gpt_cond_latent=lat,
                      speaker_embedding=emb, temperature=temperature,
                      repetition_penalty=repetition_penalty, top_k=top_k, top_p=top_p,
                      length_penalty=1.0, enable_text_splitting=split)
    wav = np.asarray(out["wav"], dtype=np.float32)
    secs = len(wav) / 24000
    if show:
        print("  %s | %.2f s audio | RTF %.2f"
              % (speaker, secs, (time.time() - t0) / max(secs, 1e-6)))

    path = os.path.join(OUT, save_as or ("%s_%03d.wav" % (speaker, len(os.listdir(OUT)))))
    sf.write(path, wav, 24000)
    if show:
        display(Audio(wav, rate=24000))
    return wav


def _ab(text, speaker, temperature, rp, top_k, top_p, split):
    """Same text, same seed, fp16 file vs fp32 file -- listen for a difference.

    Session B found none across three seeds on every objective metric. Whether a
    listener can hear one is a separate question, and this is how to ask it.
    """
    global _SLIM
    if "_SLIM" not in globals():
        print("loading the fp32 model as well (once)...")
        _SLIM, _ = load(SLIM)
    print(text, "\n  ->", to_ascii(text))
    for tag, m in (("fp16  0.934 GB", MODEL), ("fp32  1.868 GB", _SLIM)):
        torch.manual_seed(1234)
        print("\n" + tag)
        say(text, speaker, model=m, temperature=temperature, repetition_penalty=rp,
            top_k=top_k, top_p=top_p, split=split, show=False,
            save_as="ab_%s.wav" % tag.split()[0])
        display(Audio(os.path.join(OUT, "ab_%s.wav" % tag.split()[0])))


print("ready:  say('ආයුබෝවන්, ඔබට කොහොමද?')")

## 5. Edit this list and re-run - as often as you like

In [ ]:
TEXTS = [
    "ආයුබෝවන්, ඔබට කොහොමද?",
    "මට සිංහල භාෂාවෙන් කතා කරන්න පුළුවන්.",
    "අද දවස ලස්සන දවසක්. හෙට වැස්ස එයි කියලා හිතනවා.",
    "මේ ගැන නම් මට සහතික වෙන්න බැහැ, මම හරියටම දැක්කෙ නැති නිසා.",
]

for t in TEXTS:
    say(t, "dinithi")
    print("-" * 70)

## 6. The same sentences in the other voice

harini is the weaker speaker on every measured axis except SECS - 2.11 h of audio against
dinithi's 4.69 h, and F0 correlation 0.33 against 0.56. That gap has held across all five
training runs. This is where you find out whether it is audible or only statistical.

In [ ]:
for t in TEXTS:
    say(t, "harini")
    print("-" * 70)

## 7. Type freely

A text box, if `ipywidgets` is available in this session. If it is not, just call `say()`
directly in a new cell - that always works.

In [ ]:
try:
    import ipywidgets as widgets

    box = widgets.Textarea(value="ඔබට සුබ දවසක් වේවා!", rows=3,
                           layout=widgets.Layout(width="100%"))
    who = widgets.ToggleButtons(options=["dinithi", "harini"])
    temp = widgets.FloatSlider(value=0.75, min=0.5, max=0.95, step=0.05,
                               description="temp")
    go, out = widgets.Button(description="Speak", button_style="primary"), widgets.Output()

    def _go(_):
        out.clear_output()
        with out:
            say(box.value, who.value, temperature=temp.value)

    go.on_click(_go)
    display(widgets.VBox([box, who, temp, go, out]))
except ImportError:
    print("ipywidgets unavailable -- use say('...') in a cell instead")

## 8. fp16 against fp32, by ear

Session B measured no difference across three seeds: MCD 62.78 vs 63.07, F0 corr 0.438 vs
0.436, UTMOS 2.681 vs 2.677 - all inside a sampling-noise floor of +/-0.25 dB MCD and
+/-0.014 F0 corr, measured on identical weights at three seeds.

If you cannot hear a difference either, the 0.934 GB file ships and the 1.868 GB one does
not need to.

In [ ]:
say("මේ දෙක අතර වෙනසක් තියෙනවද කියලා අහලා බලන්න.", "dinithi", ab=True)

## 9. Keep the audio

In [ ]:
import os
print("%d files in %s" % (len(os.listdir(OUT)), OUT))
for f in sorted(os.listdir(OUT)):
    print("  ", f)
print("\nDownload the samples/ folder from the Output pane before the session ends.")